In [2]:
from src.eval.symantic.generation_utils import generate_answers

In [3]:
model_id = 'Hannibal046/xrag-7b'
# model_id = 'brimmann2/xgemma3-1b-v1'
dataset_id = 'brimmann2/squad_qa1'
split_name = "train"

In [4]:
generated_docs, generated_docs_embeddings, original_docs_embeddings = generate_answers(model_id, dataset_id, split_name)

`torch_dtype` is deprecated! Use `dtype` instead!


loading generator model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generator model loaded
loading embedder model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

embedder model loaded
dataset_id is not None
generating embeddings...
embeddings generated
all messages [[{'role': 'user', 'content': "Background: <xRAG> Please offer a restatement of the background sentences I've just read."}], [{'role': 'user', 'content': 'In other words, background: <xRAG> is just another way of saying:'}], [{'role': 'user', 'content': "You're getting across the same point whether you say background: <xRAG> or"}]]
prompts ["<s>[INST] Background: <xRAG> Please offer a restatement of the background sentences I've just read. [/INST]", '<s>[INST] In other words, background: <xRAG> is just another way of saying: [/INST]', "<s>[INST] You're getting across the same point whether you say background: <xRAG> or [/INST]"]
generatring documetns from embeddings...


4it [00:38,  9.62s/it]                                      


documents generated
loading embedder model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

embedder model loaded
dataset_id is None
generating embeddings...
embeddings generated


In [5]:
generated_docs

["In January 2009, President Obama announced the United States would contribute $100 million to the United Nations Population Fund (UNFPA) to support family planning and reproductive health programs in developing countries. The United States had previously suspended funding to the UNFPA in 2002 due to concerns over the organization's support for China's coercive population control policies. In February 2009, the United States announced it would resume funding to the UNFPA, with the condition that the organization would not use the funds to support China's population control policies.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>",
 "The Sex Pistols were a British punk rock 

In [4]:
generated_docs

['The United Nations has been working to promote the adoption of the "Global Health Action Plan" by the World Health Organization (WHO) and the United Nations Development Programme (UNDP) since 2008. The plan has been endorsed by 183 countries, and has been adopted by 193 countries. The plan has been supported by the United States, the European Union, and the United Kingdom.<end_of_turn><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>',
 'The band\'s early work was heavily 

In [5]:
generated_docs_embeddings

[[tensor([-1.0078,  2.2344, -0.6602,  ...,  6.8438, -0.0520,  0.7461],
         dtype=torch.bfloat16)],
 [tensor([-0.5781, -0.5977,  2.1406,  ...,  4.7188, -5.0938, -5.3125],
         dtype=torch.bfloat16)],
 [tensor([ 1.6484, -1.5000,  0.3809,  ...,  5.8438, -4.6562,  6.2188],
         dtype=torch.bfloat16)]]

In [6]:
original_docs_embeddings

[[tensor([-1.3906, -3.3125, -5.5000,  ...,  8.0625,  0.8750,  5.2500],
         dtype=torch.bfloat16)],
 [tensor([ 3.6719, -3.1406, -0.9180,  ...,  8.5625, -5.1562, -0.9766],
         dtype=torch.bfloat16)],
 [tensor([ 3.4688,  1.3828, -0.2021,  ...,  3.2344, -4.9375,  4.1875],
         dtype=torch.bfloat16)]]

In [6]:
import torch
import torch.nn.functional as F

In [7]:
stacked_embeddings_original = torch.stack([tensor_list[0] for tensor_list in original_docs_embeddings])

In [8]:
stacked_embeddings_generated = torch.stack([tensor_list[0] for tensor_list in generated_docs_embeddings])

In [9]:
stacked_embeddings_original = stacked_embeddings_original.to("cuda")
stacked_embeddings_generated = stacked_embeddings_generated.to("cuda")

In [17]:
stacked_embeddings_original

tensor([[-1.3906, -3.3125, -5.5000,  ...,  8.0625,  0.8750,  5.2500],
        [ 3.6719, -3.1406, -0.9180,  ...,  8.5625, -5.1562, -0.9766],
        [ 3.4688,  1.3828, -0.2021,  ...,  3.2344, -4.9375,  4.1875]],
       device='cuda:0', dtype=torch.bfloat16)

In [10]:
cosine_sim_batch = torch.nn.functional.cosine_similarity(
    stacked_embeddings_generated, 
    stacked_embeddings_original, 
    dim=1
)

In [19]:
cosine_sim_batch

tensor([0.6562, 0.6328, 0.6016], device='cuda:0', dtype=torch.bfloat16)

In [12]:
cosine_sim_batch

tensor([0.8477, 0.6289, 0.6680], device='cuda:0', dtype=torch.bfloat16)

In [13]:
scores_list = cosine_sim_batch.cpu().float().tolist()

In [15]:
average_score = cosine_sim_batch.mean().item()

In [17]:
data_to_save = {
    'similarity_scores': scores_list,
    'average_score': average_score
}

In [18]:
file_path = 'results/pretrain/similarity_scores.json'

In [19]:
import json

In [20]:
with open(file_path, 'w') as f:
    json.dump(data_to_save, f, indent=4)